# EDUAI Audio Lab · Mi voz con Kaggle GPU (Chatterbox Multilingual V3)

**Modo manual y privado:** enciendes el notebook cuando lo necesitas; ejecutas el modelo y descargas un WAV que puedes importar a EDUAI. No se abre ningún servidor ni túnel y no se requieren claves Supabase, Gemini o Hugging Face.

1. **Settings → Accelerator → GPU T4 x2**, y **Settings → Turn on internet**.
2. Mantén el notebook **Private**. No publiques audios, referencias ni los resultados.
3. Ejecuta las celdas en orden. La primera instalación puede tardar varios minutos.
4. Sube una muestra **de tu propia voz** de 10–30 segundos al control de archivo; la convertiremos a WAV mono 24 kHz.
5. Cambia `TEXTO`, genera y descarga `eduai_voz_kaggle.wav` desde **Output**.
6. Regresa a **EDUAI → Audio Lab → Mis voces** e importa el WAV en el perfil apropiado.

La sesión de Kaggle puede finalizar en cualquier momento. Para volver a usarlo, abre de nuevo **tu notebook guardado** desde el botón de EDUAI y vuelve a ejecutar las celdas. Las dos T4 son GPU independientes: Chatterbox usa la GPU 0, con 16 GB VRAM. Este procedimiento prueba inferencia; no marca el perfil EDUAI como «clonado/ready» en ausencia de un servicio permanente.

**Privacidad:** Kaggle es un proveedor externo que procesará temporalmente la muestra al subirla. No compartas públicamente el notebook. La exportación del resultado a EDUAI es manual, para evitar exponer credenciales de administrador. Solo usar tu voz o una voz para la que tengas autorización y consentimiento.

In [ ]:
# Comprobar GPU T4 (no usar CPU para este notebook).
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Activa Settings → Accelerator → GPU T4 x2 y reinicia la sesión.")
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# Instalar versión oficial compatible con los pesos multilingual V3.
# Solo ModelScope y GitHub oficial; NO utilizamos el descargador de HF.
import os, subprocess, sys
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "chatterbox-tts==0.1.7", "modelscope>=1.20,<3", "ipywidgets"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                "git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2"], check=True)
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
import inspect
assert "t3_model" in inspect.signature(ChatterboxMultilingualTTS.from_local).parameters, "Chatterbox V3 no quedó instalado"
print("Motor Chatterbox Multilingual V3 preparado.")

In [ ]:
# Pesos V3 desde ModelScope, no Hugging Face.
from pathlib import Path
import hashlib
MODEL_DIR = Path("/kaggle/working/eduai_chatterbox_v3")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FILES = ["ve.pt", "s3gen.pt", "t3_mtl23ls_v3.safetensors",
         "grapheme_mtl_merged_expanded_v1.json"]
missing = [name for name in FILES if not (MODEL_DIR / name).is_file()]
if missing:
    subprocess.run(["modelscope", "download", "--model", "ResembleAI/chatterbox",
                    *missing, "--local_dir", str(MODEL_DIR)], check=True)
for name in FILES:
    file = MODEL_DIR / name
    assert file.is_file() and file.stat().st_size > 2048, f"Falta: {name}"
print("Archivos verificados presentes:", len(FILES), "· modelo descargado de ModelScope")

In [ ]:
# Sube tu archivo de voz. El notebook debe permanecer PRIVADO.
import ipywidgets as widgets
from IPython.display import display
upload_voice = widgets.FileUpload(accept=".wav,.mp3,.m4a,.ogg", multiple=False)
display(upload_voice)
print("Sube tu muestra (10–30 segundos); luego ejecuta la celda siguiente.")

In [ ]:
# Normalizar la muestra y preparar la voz en GPU. No se imprime ni almacena la voz en outputs.
import wave
from pathlib import Path
entry = upload_voice.value
if not entry:
    raise ValueError("Primero sube tu muestra vocal en la celda anterior.")
item = list(entry.values())[0] if isinstance(entry, dict) else entry[0]
raw_bytes = bytes(item["content"])
if len(raw_bytes) > 12 * 1024 * 1024:
    raise ValueError("La muestra supera los 12 MB.")
original = Path("/kaggle/working/eduai_input_private")
reference = Path("/kaggle/working/eduai_reference_private.wav")
original.write_bytes(raw_bytes)
subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                "-i", str(original), "-ac", "1", "-ar", "24000",
                "-acodec", "pcm_s16le", str(reference)], check=True)
original.unlink(missing_ok=True)
with wave.open(str(reference), "rb") as w:
    duration = w.getnframes() / w.getframerate()
if not (10 <= duration <= 30):
    reference.unlink(missing_ok=True)
    raise ValueError(f"Muestra: {duration:.1f}s. Debe durar entre 10 y 30 segundos.")
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_local(ckpt_dir=MODEL_DIR, device="cuda", t3_model="v3")
model.prepare_conditionals(str(reference))
reference.unlink(missing_ok=True)
upload_voice.value = () if not isinstance(upload_voice.value, dict) else {}
print("Referencia preparada en memoria; copia temporal eliminada. Puedes generar varias frases mientras esta sesión siga activa.")

In [ ]:
# Edita aquí lo que deseas que diga tu voz.
TEXTO = "Hola. Esta es una prueba de mi voz autorizada en EDUAI. Hoy aprenderemos algo nuevo."
if not TEXTO.strip() or len(TEXTO) > 550:
    raise ValueError("Ingresa un texto de 1 a 550 caracteres por generación.")

In [ ]:
# Síntesis de voz V3 y WAV listo para importar en EDUAI.
import torchaudio
from pathlib import Path
from IPython.display import FileLink, display
out = Path("/kaggle/working/eduai_voz_kaggle.wav")
with torch.inference_mode():
    waveform = model.generate(TEXTO.strip(), language_id="es")
torchaudio.save(str(out), waveform.detach().cpu(), model.sr, encoding="PCM_S", bits_per_sample=16)
print("Generación completa:", out.name, "· WAV; descárgalo desde Output o el enlace:")
display(FileLink(str(out)))
print("Después, utiliza «Importar WAV de Kaggle» en EDUAI Mis voces.")

### Para volver a usarlo

Con la misma sesión abierta, cambia `TEXTO` y ejecuta las dos últimas celdas. Para una sesión nueva, utiliza **Abrir mi notebook de Kaggle** en EDUAI; selección T4 x2 e Internet, y repite las celdas. El modelo y la voz temporal no sobreviven al reinicio de Kaggle.

**No guardes como versión pública** un notebook con muestras vocales o archivos generados. El resultado se incorpora a EDUAI mediante carga manual autorizada.